In [ ]:
# Librerie di sistema e utilità
import os
import warnings
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Librerie per elaborazione audio
import librosa

# PyTorch
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torchvision import transforms
import timm

# Ignoriamo i warning
warnings.filterwarnings("ignore")

print("Librerie importate con successo!")
print(f"PyTorch versione: {torch.__version__}")
print(f"timm versione: {timm.__version__}")

In [ ]:
class Config:
    def __init__(self):
        # Imposta i percorsi di base in base all'ambiente
        self.COMPETITION_NAME = "birdclef-2025"
        self.BASE_DIR = f"/kaggle/input/{self.COMPETITION_NAME}"
        self.OUTPUT_DIR = "/kaggle/working"
        self.MODELS_DIR = "/kaggle/input"  # Per i modelli pre-addestrati
            
        # Imposta subito i percorsi derivati per l'ambiente Kaggle
        self._setup_derived_paths()
        
        # Parametri per il preprocessing audio
        self.SR = 32000      # Sample rate
        self.DURATION = 5    # Durata dei clip in secondi
        self.N_MELS = 224    # Numero di bande Mel
        self.N_FFT = 2048    # Dimensione finestra FFT
        self.HOP_LENGTH = 512  # Hop length per STFT
        self.FMIN = 48       # Frequenza minima per lo spettrogramma Mel
        self.FMAX = 16000    # Frequenza massima
        self.POWER = 2       # Esponente per calcolo spettrogramma
            
        # Parametri per il device
        self.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Parametri per inference/submission
        self.TEST_CLIP_DURATION = 5  # Durata dei segmenti per la predizione (secondi)
        self.N_CLASSES = 0  # Sarà impostato dopo aver caricato i dati

    def _setup_derived_paths(self):
        """Imposta i percorsi derivati basati su BASE_DIR"""
        self.TRAIN_AUDIO_DIR = os.path.join(self.BASE_DIR, "train_audio")
        self.TEST_SOUNDSCAPES_DIR = os.path.join(self.BASE_DIR, "test_soundscapes")
        self.TRAIN_CSV_PATH = os.path.join(self.BASE_DIR, "train.csv")
        self.TAXONOMY_CSV_PATH = os.path.join(self.BASE_DIR, "taxonomy.csv") 
        self.SAMPLE_SUB_PATH = os.path.join(self.BASE_DIR, "sample_submission.csv")

# Inizializza la configurazione
config = Config()

print(f"Device utilizzato: {config.DEVICE}")
print(f"Directory Test Soundscapes: {config.TEST_SOUNDSCAPES_DIR}")
print(f"Path Sample Submission: {config.SAMPLE_SUB_PATH}")

In [ ]:
def load_metadata():
    """
    Carica e prepara i metadati dal file CSV di training.
    
    Returns:
        tuple: all_species
    """
    print(f"Caricamento metadati da: {config.TRAIN_CSV_PATH}")
    train_df = pd.read_csv(config.TRAIN_CSV_PATH)
    sample_sub_df = pd.read_csv(config.SAMPLE_SUB_PATH)
    
    # Estrai tutte le etichette uniche
    train_primary_labels = train_df['primary_label'].unique()
    train_secondary_labels = set([lbl for sublist in train_df['secondary_labels'].apply(eval) 
                                 for lbl in sublist if lbl])
    submission_species = sample_sub_df.columns[1:].tolist()  # Escludi row_id
    
    # Combina tutte le possibili etichette
    all_species = sorted(list(set(train_primary_labels) | train_secondary_labels | set(submission_species)))
    N_CLASSES = len(all_species)
    config.N_CLASSES = N_CLASSES  # Aggiorna il numero di classi nella configurazione
    
    print(f"Numero totale di specie trovate: {N_CLASSES}")
    
    return all_species

# Carica le specie
all_species = load_metadata()

In [ ]:
# Crea una singola istanza della trasformazione MelSpectrogram da riutilizzare
mel_transform = T.MelSpectrogram(
    sample_rate=config.SR,
    n_fft=config.N_FFT,
    win_length=None,
    hop_length=config.HOP_LENGTH,
    f_min=config.FMIN,
    f_max=config.FMAX,
    n_mels=config.N_MELS,
    window_fn=torch.hann_window,
    power=config.POWER,
    normalized=False,
    onesided=True,
    norm="slaney",
    mel_scale="slaney"
)

# Funzione di conversione a dB e normalizzazione
def amplitude_to_db(spectrogram):
    """Converti spettrogramma in scala dB e normalizza tra 0-1"""
    # Converti in dB
    spectrogram_db = 10.0 * torch.log10(torch.clamp(spectrogram, min=1e-10))
    
    # Normalizza
    min_val = torch.min(spectrogram_db)
    max_val = torch.max(spectrogram_db)
    if max_val > min_val:
        return (spectrogram_db - min_val) / (max_val - min_val)
    else:
        return torch.zeros_like(spectrogram_db)

In [ ]:
class EfficientNetBirdClassifier(nn.Module):
    def __init__(self, num_classes=config.N_CLASSES, pretrained=False, model_name='efficientnet_b0'):
        super(EfficientNetBirdClassifier, self).__init__()
        
        # Crea il modello
        self.efficientnet = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0  # Rimuovi il classificatore originale
        )
        
        # Ottieni la dimensione dell'output del feature extractor
        if hasattr(self.efficientnet, 'num_features'):
            classifier_in_features = self.efficientnet.num_features
        elif hasattr(self.efficientnet, 'classifier'):
            classifier_in_features = self.efficientnet.classifier.in_features
        else:
            # Valore predefinito per EfficientNet-B0
            classifier_in_features = 1280
        
        # Sostituisci il classificatore semplice con una MLP con dropout
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),  # Primo dropout significativo
            nn.Linear(classifier_in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),  # Secondo dropout più leggero
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        # Se l'input è un'immagine a 1 canale, replicala su 3 canali
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)
        
        # Passa l'input attraverso il backbone per ottenere le features
        features = self.efficientnet(x)
        
        # Passa le feature attraverso il classificatore
        output = self.classifier(features)
        
        return output

In [ ]:
# Percorso del modello pre-addestrato (modifica secondo necessità)
model_path = "/kaggle/input/mio-modello-addestrato/birdclef_efficientNET_dataAugPaper_timm_best.pth"

# Inizializza il modello
model = EfficientNetBirdClassifier(num_classes=config.N_CLASSES, pretrained=False).to(config.DEVICE)

# Carica il modello pre-addestrato
try:
    print(f"Caricamento del modello pre-addestrato da {model_path}...")
    checkpoint = torch.load(model_path, map_location=config.DEVICE)
    
    # Verifica se è un dict con model_state_dict o direttamente state_dict
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Modello caricato con successo (epoca: {checkpoint.get('epoch', 'N/A')})")
    else:
        model.load_state_dict(checkpoint)
        print("Modello caricato con successo")
    
    model.eval()  # Imposta il modello in modalità valutazione
except Exception as e:
    print(f"Errore nel caricamento del modello: {e}")
    raise

In [ ]:
train_soundscapes_dir = "/kaggle/input/birdclef-2025/train_soundscapes"
output_dir = "/kaggle/working/pseudo_labeled_data"

In [ ]:
def simplified_pseudo_labelling(model, soundscapes_dir, device=config.DEVICE, threshold=0.5):
    """
    Strategia semplificata di pseudo-labelling che massimizza la copertura mantenendo 
    un buon livello di qualità.
    """
    model.to(device)
    model.eval()
    
    results = []
    
    # Lista di tutti i file audio
    audio_files = [f for f in os.listdir(soundscapes_dir) if f.endswith('.ogg')]
    print(f"Trovati {len(audio_files)} file audio da etichettare")
    
    for audio_file in tqdm(audio_files, desc="Processing soundscapes"):
        # Carica il file audio
        audio_path = os.path.join(soundscapes_dir, audio_file)
        try:
            signal, sr = librosa.load(audio_path, sr=config.SR)
        except Exception as e:
            print(f"Errore caricamento {audio_file}: {e}")
            continue
            
        # Estrai solo 3 segmenti strategici (inizio, centro, fine)
        segments = []
        segment_length = config.SR * config.DURATION
        
        if len(signal) >= segment_length * 3:  # File abbastanza lungo
            # Segmento iniziale (25% della registrazione)
            start_idx = int(len(signal) * 0.25) - segment_length//2
            start_idx = max(0, start_idx)
            segments.append(signal[start_idx:start_idx + segment_length])
            
            # Segmento centrale
            mid_idx = len(signal)//2 - segment_length//2
            segments.append(signal[mid_idx:mid_idx + segment_length])
            
            # Segmento finale (75% della registrazione)
            end_idx = int(len(signal) * 0.75) - segment_length//2
            end_idx = min(len(signal) - segment_length, end_idx)
            end_idx = max(0, end_idx)  # Assicura che non sia negativo
            segments.append(signal[end_idx:end_idx + segment_length])
        else:
            # Audio troppo corto, usa tutto e padda se necessario
            if len(signal) < segment_length:
                signal = np.pad(signal, (0, segment_length - len(signal)), mode='constant')
            segments.append(signal)
        
        # Fai predizioni sui segmenti
        all_predictions = []
        
        for segment in segments:
            # Calcola spettrogramma Mel
            mel_spec = librosa.feature.melspectrogram(
                y=segment, 
                sr=config.SR,
                n_fft=config.N_FFT,
                hop_length=config.HOP_LENGTH,
                n_mels=config.N_MELS,
                fmin=config.FMIN,
                fmax=config.FMAX
            )
            
            # Converti in scala logaritmica (dB) e normalizza
            log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
            min_val = np.min(log_mel_spec)
            max_val = np.max(log_mel_spec)
            if max_val > min_val:
                log_mel_spec = (log_mel_spec - min_val) / (max_val - min_val)
            else:
                log_mel_spec = np.zeros_like(log_mel_spec)
            
            # Prepara il tensor per il modello
            log_mel_spec = np.expand_dims(np.expand_dims(log_mel_spec, axis=0), axis=0)
            input_tensor = torch.tensor(log_mel_spec, dtype=torch.float32).to(device)
            
            # Resize a 224x224 come nel training
            resize_transform = transforms.Resize((224, 224), 
                interpolation=transforms.InterpolationMode.BICUBIC)
            input_tensor = resize_transform(input_tensor)

            # Effettua predizione
            with torch.no_grad():
                output = model(input_tensor)
                scores = torch.sigmoid(output).cpu().numpy()[0]
                
                # Pulisci i valori bassi (key step menzionato)
                scores = np.where(scores < 0.1, 0, scores)
                
                all_predictions.append(scores)
        
        # Ottieni la classe più probabile per ogni segmento
        top_classes = [np.argmax(pred) for pred in all_predictions]
        
        # Strategia di votazione: conta quante volte appare ogni classe
        from collections import Counter
        votes = Counter(top_classes)
        
        # Se c'è un vincitore chiaro (2 o più voti su 3 segmenti)
        if votes.most_common(1)[0][1] >= 2:
            best_class_idx = votes.most_common(1)[0][0]
            predicted_class = all_species[best_class_idx]
            
            # Ottieni la confidenza media per questa classe nei segmenti
            confidences = [pred[best_class_idx] for pred in all_predictions]
            avg_confidence = np.mean(confidences)
            
            # Accetta solo se la confidenza media è sopra la soglia
            if avg_confidence > threshold:
                # Calcola la "soft label" (media delle probabilità)
                soft_labels = np.mean(all_predictions, axis=0)
                
                results.append({
                    'filename': audio_file,
                    'predicted_label': predicted_class,  
                    'confidence': avg_confidence,
                    'soft_labels': soft_labels  # Aggiungi le soft labels
                })
        else:
            # Se non c'è un vincitore chiaro, prendi la classe con la confidenza più alta
            max_conf_per_class = {}
            for class_idx in range(len(all_species)):
                confidences = [pred[class_idx] for pred in all_predictions]
                max_conf_per_class[class_idx] = max(confidences)
            
            # Trova la classe con la confidenza massima
            best_class_idx = max(max_conf_per_class, key=max_conf_per_class.get)
            max_confidence = max_conf_per_class[best_class_idx]
            predicted_class = all_species[best_class_idx]
            
            # Accetta solo se la confidenza è abbastanza alta
            if max_confidence > threshold:
                # Calcola la "soft label" (media delle probabilità)
                soft_labels = np.mean(all_predictions, axis=0)
                
                results.append({
                    'filename': audio_file,
                    'predicted_label': predicted_class,
                    'confidence': max_confidence,
                    'soft_labels': soft_labels  # Aggiungi le soft labels
                })
    
    # Crea DataFrame
    results_df = pd.DataFrame(results)
    
    print(f"\nRisultati pseudo-labelling:")
    print(f"- File totali: {len(audio_files)}")
    print(f"- File etichettati: {len(results_df)} ({len(results_df)/len(audio_files):.1%})")
    
    if len(results_df) > 0:
        # Distribuzioni di classi (top 10)
        print(f"\nDistribuzione classi (top 10):")
        class_counts = results_df['predicted_label'].value_counts()
        for label, count in class_counts.head(10).items():
            print(f"  {label}: {count} file ({count/len(results_df):.1%})")
    
    return results_df

In [ ]:
def prepare_pseudo_data_with_soft_labels(labeled_df, train_soundscapes_dir, output_dir):
    """
    Prepara i dati pseudo-etichettati includendo le soft labels
    """
    import os
    import pandas as pd
    import numpy as np
    import shutil
    
    # Crea directory di output
    os.makedirs(output_dir, exist_ok=True)
    pseudo_audio_dir = os.path.join(output_dir, "pseudo_audio")
    os.makedirs(pseudo_audio_dir, exist_ok=True)
    
    # Crea directory per ogni specie
    for species in labeled_df['predicted_label'].unique():
        os.makedirs(os.path.join(pseudo_audio_dir, species), exist_ok=True)
    
    # Liste per memorizzare le righe del CSV e le soft labels
    pseudo_train_rows = []
    soft_labels_dict = {}
    
    # Processa ogni file etichettato
    for idx, row in tqdm(labeled_df.iterrows(), total=len(labeled_df)):
        # Percorsi file
        src_path = os.path.join(train_soundscapes_dir, row['filename'])
        new_filename = f"pseudo_{row['filename']}"
        species_dir = os.path.join(pseudo_audio_dir, row['predicted_label'])
        dst_path = os.path.join(species_dir, new_filename)
        
        # Copia il file audio
        shutil.copy2(src_path, dst_path)
        
        # Percorso relativo per il CSV
        rel_path = f"{row['predicted_label']}/{new_filename}"
        
        # Crea riga CSV
        new_row = {
            'primary_label': row['predicted_label'],
            'secondary_labels': '[]',  # Lista vuota
            'filename': rel_path,
            'rating': 5,  # Rating alto per pseudo-labels con buona confidenza
        }
        
        pseudo_train_rows.append(new_row)
        
        # Memorizza le soft labels
        if 'soft_labels' in row:
            soft_labels_dict[rel_path] = row['soft_labels']
    
    # Crea DataFrame e salva come CSV
    pseudo_train_df = pd.DataFrame(pseudo_train_rows)
    csv_path = os.path.join(output_dir, "pseudo_train.csv")
    pseudo_train_df.to_csv(csv_path, index=False)
    
    # Salva le soft labels
    np.save(os.path.join(output_dir, "soft_labels.npy"), soft_labels_dict)
    
    print(f"\nDati pseudo-etichettati preparati:")
    print(f"- File CSV: {csv_path}")
    print(f"- Soft labels: {os.path.join(output_dir, 'soft_labels.npy')}")
    print(f"- File audio: {pseudo_audio_dir}")
    print(f"- Totale esempi: {len(pseudo_train_df)}")
    
    return pseudo_train_df, soft_labels_dict

In [ ]:
labeled_df = simplified_pseudo_labelling(
    model=model,
    soundscapes_dir=train_soundscapes_dir,
    threshold=0.5  # Usa un valore più basso per aumentare la copertura
)

# Aggiungi una riga vuota per skipped_df (se ancora necessaria)
skipped_df = pd.DataFrame()

# Salva solo labeled_df
labeled_df.to_csv("/kaggle/working/labeled_soundscapes.csv", index=False)

# 4. Prepara i dati in formato train.csv e salva i file
pseudo_train_df = prepare_pseudo_data_with_soft_labels(
    labeled_df=labeled_df,
    train_soundscapes_dir=train_soundscapes_dir,
    output_dir=output_dir
)